# Satellite fold: continuation in $\varepsilon_d/g_2$ and comparison with the analytic existence boundary (20/10/2026)

Report v2, big error 1 ("to do"). `Hill_method_cat_fold.ipynb` showed that the satellite is born at a fold, a saddle-node of limit cycles.
At $\varepsilon_d = 10.12\,g_2$ this fold is at $g_2/2\pi = 40.4$ MHz, and the satellite and the saddle are created together there.
The saddle then joins the cat at its own fold, at 97.2 MHz. This notebook:
1. continues the satellite fold in $\varepsilon_d/g_2$, with Hill's method and pseudo-arclength continuation;
2. compares it with the **analytic existence boundary** used for the seeds of the satellite-start map (`satellite_hb` in `lyapunov_heatmap_satellite_start.ipynb`: App. "Satellites" at quadratic order in the junction phase, $\sigma = +1$, with the phase-locking condition $2\kappa_b'|y| \le \varepsilon_d'$), with the reduced system of the appendix kept at quadratic order and with the full cosine, and with a leading-order closed form;
3. checks the dependence on $\kappa_b$;
4. explains the band of the satellite-start map where the satellite exists but the trajectory ends on a cat;
5. (22/10/2026) checks whether the single period-1 branch beyond the cusp period-doubles or undergoes a Neimark–Sacker bifurcation at large $g_2$.

The conventions are those of the other Hill notebooks: $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\kappa_b = \pi/10.4$ in the code convention, $C = \mathrm{diag}(0, \kappa_b/\omega_b)$, $\tau = \omega_a t$.
The continuation parameter is $s = \sin\varepsilon_p$, so that $g_2 = s\,E_J\varphi_a^2\varphi_b$.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from scipy.special import jv
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/"

w_a = 25.338776456203686
w_b = 2*w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4                 # 1/ns, code convention (phi_b'' = ... - 2 kappa_b phi_b')
G0 = E_J*phi_a**2*phi_b              # g2 = s G0, s = sin(eps_p)
n = 2
M = np.diag([1.0, 0.25])
K = np.eye(2)

def MHz(g):
    return g/2/np.pi*1e3

## Harmonic balance (exact one-period AFT, as in `Hill_method_cat_fold.ipynb`)
`HB(N_H, kfac)` uses $\kappa_b \to$ `kfac`$\,\kappa_b$. `set_s(s, r)` sets $g_2 = sG_0$ and $\varepsilon_d = r\,g_2$.

In [ ]:
class HB:
    def __init__(self, N_H, kfac=1.0, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.C = np.diag([0.0, kfac*kappa_b/w_b])
        dim = n*(2*N_H + 1)
        self.L = np.zeros((dim, dim))
        self.L[:n, :n] = K
        for k in range(1, N_H + 1):
            A, Bk = K - k**2*M, k*self.C
            s_, c_ = 2*k - 1, 2*k
            self.L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            self.L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            self.L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            self.L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
        t = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        k = np.arange(1, N_H + 1)
        Phi = np.empty((self.N, 2*N_H + 1))
        Phi[:, 0] = 1/np.sqrt(2)
        Phi[:, 1::2] = np.sin(np.outer(t, k))
        Phi[:, 2::2] = np.cos(np.outer(t, k))
        self.Gamma = np.kron(Phi, np.eye(n))
        self.Gamma_pinv = (2/self.N)*self.Gamma.T

    def set_s(self, s, r):
        self.B = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])          # (beta_a', beta_b')
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*2*n + 1] = -np.sqrt(2)*r*s*G0*phi_b/w_b             # c_2 block of mode b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        sn = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*sn, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol, options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(sol.message)
        return sol.x


def orbit(z, N_H, t):
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s_, c_ = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s_ + np.cos(np.outer(t, k)) @ c_
        v = np.cos(np.outer(t, k)) @ (k*s_) - np.sin(np.outer(t, k)) @ (k*c_)
        out.append((x, v))
    return out

def strobe(z, N_H, tau=0.0):
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

def memory_phase(z):
    # phase of c1 + i s1 of the memory (cat: +-1/2, satellite: ~0 or ~1, in units of pi)
    za = z[::2]
    return np.angle(za[2] + 1j*za[1])/np.pi

def max_abs_mu(hb, z):
    # monodromy over one period (variational equation around the HB orbit)
    Minv, base = np.linalg.inv(M), np.outer(hb.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hb.N_H, np.atleast_1d(t))
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hb.C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.max(np.abs(np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))))

def full_rhs(s, r, kfac=1.0):
    beta = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p, kp = r*s*G0*phi_b/w_b, kfac*kappa_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        return [v_a, v_b, -x_a + np.sqrt(2)*beta[0]*g, 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - kp*v_b)]
    return f

## Analytic satellite (port of `satellite_hb`)
This is the quadratic-order system of App. "Satellites" with $\sigma = +1$: the scalar equation for $\bar y$, then $x^2 > 0$, then the phase locking $2\kappa_b'|y| \le \varepsilon_d'$.
$\kappa_b' = \kappa_b/\omega_b$ in the code convention, which is $\kappa_b/(2\omega_b)$ in the article's GKSL convention.
The function returns the rotating-frame amplitudes in the paper's units ($a_\text{paper} = \varphi_a a$), or `None`.
`seed_z` converts them into HB coefficients, and `seed_state` into the laboratory state at $\tau = 0$ (the seed of the satellite-start map).

In [ ]:
yb_grid = np.linspace(1e-3, 3.0, 3000)

def satellite_hb(s, r, kfac=1.0):
    kp = kfac*kappa_b/w_b
    ba, bb = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    ed, bS = r*s*G0*phi_b/w_b, ba + bb
    gam = 1 + bb/(ba*(1 + kp**2))
    eeff = lambda y: np.sqrt(np.maximum(ed**2 - 4*kp**2*y**2, 0.0))
    def y_of(yb):
        y = yb + ba*ed/(2*bb)
        for _ in range(30):
            y = yb + ba*eeff(y)/(2*bb)
        return y
    def u_of(yb):
        eta = 2*bb*yb
        return (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)
    def x2_of(yb):
        u = u_of(yb)
        return (2*(1 + u)*yb**2 - eeff(y_of(yb))/bb)/(1 + 2*u)
    def resid(yb):
        u = u_of(yb)
        return yb*(1 + u) - 2*bS*((1 + (1 + u)**2/2)*yb**2 + (1 + u**2)*x2_of(yb))
    rv = resid(yb_grid)
    for k in np.where(rv[:-1]*rv[1:] <= 0)[0]:          # first physical root, refined by bisection
        lo, hi = yb_grid[k], yb_grid[k + 1]
        for _ in range(60):
            mid = (lo + hi)/2
            if resid(lo)*resid(mid) <= 0:
                hi = mid
            else:
                lo = mid
        yb = (lo + hi)/2
        x2 = x2_of(yb)
        if x2 <= 0:
            continue
        u, y, x = u_of(yb), y_of(yb), np.sqrt(x2)
        if 2*kp*abs(y) > ed:
            continue
        th = -np.arcsin(2*kp*y/ed)/2
        a1 = -(1 + u)*yb/(2*gam)
        return dict(a0=x*np.exp(1j*th), a1=a1, am1=-ba*eeff(y)/(2*bb)*np.exp(2j*th),
                    b0=y*np.exp(2j*th), bm1=u*x*np.exp(3j*th), b2=bb/ba*a1)
    return None

def seed_z(S, N_H):
    # x^lambda_a = sqrt2 Re(a_paper e^{-i tau}), x^lambda_b = sqrt2 Re(b_paper e^{-2 i tau})
    za, zb = np.zeros(2*N_H + 1), np.zeros(2*N_H + 1)
    za[0] = 2*np.real(S["a1"])
    za[1], za[2] = np.sqrt(2)*np.imag(S["a0"]), np.sqrt(2)*np.real(S["a0"])
    za[3], za[4] = np.sqrt(2)*np.imag(S["am1"]), np.sqrt(2)*np.real(S["am1"])
    zb[0] = 2*np.real(S["b2"])
    zb[3], zb[4] = np.sqrt(2)*np.imag(S["b0"]), np.sqrt(2)*np.real(S["b0"])
    zb[5], zb[6] = np.sqrt(2)*np.imag(S["bm1"]), np.sqrt(2)*np.real(S["bm1"])
    z = np.zeros(n*(2*N_H + 1)); z[::2], z[1::2] = za, zb
    return z

def seed_state(S):
    a = S["a0"] + S["a1"] + S["am1"]; b = S["b0"] + S["bm1"] + S["b2"]
    return np.array([np.sqrt(2)*a.real, np.sqrt(2)*b.real, np.sqrt(2)*a.imag, 2*np.sqrt(2)*b.imag])

def analytic_boundary(r, kfac=1.0, g_lo=5.0, g_hi=200.0):
    # lowest g2 (MHz) at which satellite_hb finds a satellite: coarse scan, then bisection
    gs = np.arange(g_lo, g_hi, 0.5)
    ex = np.array([satellite_hb(g/MHz(G0), r, kfac) is not None for g in gs])
    if not ex.any():
        return np.nan
    i = int(np.argmax(ex)); lo, hi = gs[max(i - 1, 0)], gs[i]
    for _ in range(25):
        mid = (lo + hi)/2
        if satellite_hb(mid/MHz(G0), r, kfac) is not None:
            hi = mid
        else:
            lo = mid
    return hi

## Exact satellite at $\varepsilon_d = 10.12\,g_2$, from the analytic seed
At 60 MHz the analytic satellite is close enough for Powell to converge directly. The HB satellite is the partner $a\to-a$ of the one followed in `Hill_method_cat_fold`, at the same $|a|$.

In [ ]:
N_H = 16
r0, g_seed = 10.12, 60.0
hb = HB(N_H)
s_seed = g_seed/MHz(G0)
hb.set_s(s_seed, r0)
z_sat60 = hb.solve(seed_z(satellite_hb(s_seed, r0), N_H))
print(f"satellite at {g_seed} MHz: memory phase {memory_phase(z_sat60):+.3f} pi, "
      f"|a_1| = {np.hypot(z_sat60[::2][1], z_sat60[::2][2])/np.sqrt(2)/phi_a:.2f}, max|mu| = {max_abs_mu(hb, z_sat60):.4f}")

## Fold of the HB satellite, continued in $\varepsilon_d/g_2$
For each $\varepsilon_d/g_2$, a satellite point about 12% above the previous fold is continued in the ratio at fixed $s$; if it is lost, $s$ is raised first.
Pseudo-arclength in $(z, s)$, towards decreasing $s$, then runs until $s$ turns back: that turning point is the fold. The fold is refined by a parabola through the last three points.

In [ ]:
def continue_to_fold(hb, z, s, r, ds=0.002, nmax=4000, direction=-1):
    def F(y):
        hb.set_s(y[-1], r); return hb.residual(y[:-1])
    def dF(y):
        hb.set_s(y[-1], r); Jz, r_ = hb.jacobian(y[:-1]), hb.residual(y[:-1])
        h = 1e-7
        hb.set_s(y[-1] + h, r)
        return np.column_stack([Jz, (hb.residual(y[:-1]) - r_)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dF(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and np.sign(t[-1]) != direction):
            t = -t
        return t
    y = np.append(z, s); t = tangent(y); hist = [y.copy()]
    for _ in range(nmax):
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        if not sol.success:
            ds /= 2
            if ds < 1e-7:
                break
            continue
        y = sol.x; t_new = tangent(y, t); hist.append(y.copy())
        if np.sign(t_new[-1]) != np.sign(t[-1]):
            c = np.polyfit([-1, 0, 1], np.array(hist)[-3:, -1], 2)
            return np.polyval(c, -c[1]/(2*c[0])), np.array(hist)
        t = t_new
    return np.nan, np.array(hist)

ratio_list = np.round(np.arange(2.0, 13.51, 0.5), 2)
sat_fold = {}                                         # ratio -> (s at the fold, memory phase at the fold)
for part in (ratio_list[ratio_list >= r0], ratio_list[ratio_list < r0][::-1]):
    z, s, r_prev = z_sat60.copy(), s_seed, r0
    for r in part:
        for attempt in range(40):
            try:
                zz = z.copy()
                for rr in np.linspace(r_prev, r, 6)[1:]:
                    hb.set_s(s, rr); zz = hb.solve(zz)
                break
            except RuntimeError:
                hb.set_s(s*1.03, r_prev); z = hb.solve(z); s *= 1.03
        s_f, hist = continue_to_fold(hb, zz, s, r)
        pre = hist[hist[:, -1] >= s_f]                 # satellite side of the fold
        j = np.argmin(np.abs(pre[:, -1] - 1.12*s_f))
        z, s, r_prev = pre[j, :-1], pre[j, -1], r
        sat_fold[r] = (s_f, memory_phase(hist[-1, :-1]))
g2_sat_fold = np.array([MHz(sat_fold[r][0]*G0) for r in ratio_list])
for r in ratio_list[::2]:
    print(f"eps_d/g2 = {r:5.2f}: satellite fold at g2/2pi = {MHz(sat_fold[r][0]*G0):6.2f} MHz, memory phase {sat_fold[r][1]:.3f} pi")

## Reduced system of App. "Satellites" with the phase $\theta$ as unknown
Eq. (app_sat_bessel) eliminates $\theta$ through $\sigma = \mathrm{sign}\cos2\theta$ and $\varepsilon_\text{eff}$. Keeping $\theta$ as an unknown, with the two drive equations
$2\beta_b'R_{-2} = \varepsilon_d'\cos2\theta$ and $\varepsilon_d'\sin2\theta = -2\kappa_b'y$, makes the system smooth through $\cos 2\theta = 0$, i.e. through $\varepsilon_\text{eff} = 0$.
That is where the cat ($\sigma = -1$) and satellite ($\sigma = +1$) families connect, so this system has both folds of the S-shaped branch.
The unknowns are $(x, \bar y, w, C, y, \theta)$. `quad=True` uses $\cos X - 1 \simeq -X^2/2$ (Eq. app_sat_poly), `quad=False` the full cosine (Bessel sums truncated at $|m|, |p| \le 8$).
The satellite fold is found by starting on the satellite ($\theta \approx \pi$) well above the fold and continuing towards lower $g_2$.

In [ ]:
m_, p_ = np.meshgrid(np.arange(-8, 9), np.arange(-8, 9), indexing="ij")

def R_N(N, x, yb, w, C, quad):
    if quad:
        return {-1: -x*(C + yb) - yb*w, 0: 1 - (C**2/2 + x**2 + yb**2 + w**2),
                -2: -(C*yb + x**2/2 + x*w), -3: -(C*w + x*yb)}[N]
    return np.sum(jv(N - 2*m_ - 3*p_, 2*x)*jv(m_, 2*yb)*jv(p_, 2*w)*np.cos(C + (N - m_ - 2*p_)*np.pi/2))

def reduced_residual(v, s, r, quad=False, kfac=1.0):
    x, yb, w, C, y, th = v
    kp = kfac*kappa_b/w_b
    ba, bb = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    ed = r*s*G0*phi_b/w_b
    gam = 1 + bb/ba if quad else 1 + bb/(ba*(1 + kp**2))
    R = lambda N: R_N(N, x, yb, w, C, quad)
    return np.array([R(-1), C - 2*gam*ba*(R(0) - 1),
                     2*bb*R(-2) - ed*np.cos(2*th), ed*np.sin(2*th) + 2*kp*y,
                     w + 2*bb*R(-3), y - yb - ba*R(-2)])

def reduced_satellite_fold(r, g_start, quad=False, kfac=1.0):
    s0 = g_start/MHz(G0)
    ba, bb = 2*E_J*s0*phi_a**2/w_a, 2*E_J*s0*phi_b**2/w_b
    bS, A = ba + bb, r*phi_a**2/2
    yb = (1 + np.sqrt(1 + 56*bS**2*A))/(14*bS)                    # leading-order satellite (Eq. hb_analytic_simple)
    guess = [np.sqrt(max(2*yb**2 - A, 1e-4)), yb, 0.0, -yb, yb - ba*(r*s0*G0*phi_b/w_b)/(2*bb), np.pi]
    sol = root(lambda v: reduced_residual(v, s0, r, quad, kfac), guess, tol=1e-13)
    if not sol.success or np.cos(2*sol.x[5]) < 0:
        return np.nan, np.nan
    G = lambda Y: reduced_residual(Y[:-1], Y[-1], r, quad, kfac)
    def dG(Y, h=1e-8):
        g0 = G(Y); return np.column_stack([(G(Y + h*e) - g0)/h for e in np.eye(7)])
    Y = np.append(sol.x, s0)
    t = np.linalg.svd(dG(Y))[2][-1]; t = t if t[-1] < 0 else -t
    ds, hist = 0.005, [Y.copy()]
    for _ in range(5000):
        S = root(lambda Z: np.append(G(Z), t @ (Z - Y) - ds), Y + ds*t, tol=1e-13)
        if not S.success:
            ds /= 2
            if ds < 1e-9:
                return np.nan, np.nan
            continue
        Y = S.x
        t_new = np.linalg.svd(dG(Y))[2][-1]; t_new = t_new if t_new @ t > 0 else -t_new
        hist.append(Y.copy())
        if np.sign(t_new[-1]) != np.sign(t[-1]):
            c = np.polyfit([-1, 0, 1], np.array(hist)[-3:, -1], 2)
            return np.polyval(c, -c[1]/(2*c[0])), Y[5]/np.pi
        t = t_new; ds = min(1.3*ds, 0.02)
        if Y[-1] < 0.01:
            break
    return np.nan, np.nan

def closed_form_MHz(r, kfac=1.0):
    # leading order (u = 0) at eps_eff = 0: ybar = 1/(7 beta_Sigma) and eps_d' = 2 kappa_b' ybar
    #   -> g2 |alpha| = phi_a sqrt(kappa_b w_a / (7 (2 phi_a^2 + phi_b^2)))   (code kappa_b; = kappa_b,GKSL/14 inside the root)
    return MHz(phi_a*np.sqrt(kfac*kappa_b*w_a/(7*(2*phi_a**2 + phi_b**2)))/np.sqrt(r/2))

red_full, red_quad, th_full, analytic = [], [], [], []
for r, g in zip(ratio_list, g2_sat_fold):
    sf, th = reduced_satellite_fold(r, 1.6*g, quad=False)
    sq, _ = reduced_satellite_fold(r, 1.6*g, quad=True)
    red_full.append(MHz(sf*G0)); th_full.append(th); red_quad.append(MHz(sq*G0))
    analytic.append(analytic_boundary(r, g_lo=0.5*MHz(sq*G0)))
red_full, red_quad, analytic = map(np.array, (red_full, red_quad, analytic))
closed = closed_form_MHz(ratio_list)

print(" eps_d/g2 |  Hill  | reduced, full cos (theta/pi) | reduced, quad | analytic boundary | closed form   (MHz)")
for i in range(0, len(ratio_list), 2):
    print(f"  {ratio_list[i]:6.2f} | {g2_sat_fold[i]:6.2f} | {red_full[i]:6.2f} ({th_full[i]:.3f})            | {red_quad[i]:6.2f}        | {analytic[i]:6.2f}            | {closed[i]:6.2f}")
print(f"max |analytic boundary - reduced quad| = {np.nanmax(np.abs(analytic - red_quad)):.3f} MHz")
for name, line in (("reduced, full cos", red_full), ("reduced, quad / analytic", red_quad), ("closed form", closed)):
    rel = (line - g2_sat_fold)/g2_sat_fold
    print(f"{name:25s}: {100*np.nanmin(rel):+.1f}% .. {100*np.nanmax(rel):+.1f}% from Hill")
print("g2* |alpha| (MHz): Hill", np.round(np.nanmin(g2_sat_fold*np.sqrt(ratio_list/2)), 1), "..", np.round(np.nanmax(g2_sat_fold*np.sqrt(ratio_list/2)), 1),
      "  quad", np.round(np.nanmin(red_quad*np.sqrt(ratio_list/2)), 1), "..", np.round(np.nanmax(red_quad*np.sqrt(ratio_list/2)), 1),
      "  closed form", np.round(closed[0]*np.sqrt(ratio_list[0]/2), 1))

## Dependence on $\kappa_b$ at $\varepsilon_d = 10.12\,g_2$
The closed form predicts $g_2^\ast \propto \sqrt{\kappa_b}$. Here the HB fold is recomputed with $\kappa_b \to 0.5\,\kappa_b$ and $2\,\kappa_b$, starting from the 60 MHz satellite, together with the reduced system.

In [ ]:
kappa_check = {}
for kfac in (0.5, 1.0, 2.0):
    hbk = HB(N_H, kfac)
    zk = z_sat60.copy()
    for s_new in np.linspace(s_seed, 1.3*s_seed, 10):      # up to 78 MHz: above the fold for the three values
        hbk.set_s(s_new, r0); zk = hbk.solve(zk)
    s_f, _ = continue_to_fold(hbk, zk, s_seed*1.3, r0)
    sf_red, _ = reduced_satellite_fold(r0, 1.6*MHz(s_f*G0), quad=False, kfac=kfac)
    sq_red, _ = reduced_satellite_fold(r0, 1.6*MHz(s_f*G0), quad=True, kfac=kfac)
    kappa_check[kfac] = (MHz(s_f*G0), MHz(sf_red*G0), MHz(sq_red*G0), closed_form_MHz(r0, kfac))
    print(f"kappa_b x {kfac}: Hill {kappa_check[kfac][0]:6.2f}  reduced full {kappa_check[kfac][1]:6.2f}  "
          f"reduced quad {kappa_check[kfac][2]:6.2f}  closed form {kappa_check[kfac][3]:6.2f}  (MHz)")
k = np.array(sorted(kappa_check))
for j, name in enumerate(("Hill", "reduced full", "reduced quad", "closed form")):
    v = np.array([kappa_check[kk][j] for kk in k])
    print(f"{name:13s}: exponent d ln g2*/d ln kappa_b = {np.polyfit(np.log(k), np.log(v), 1)[0]:.2f}")

## Targeting the cusp: where the S disappears as $\kappa_b$ grows ($\varepsilon_d = 10.12\,g_2$)
- The satellite fold rises with $\kappa_b$ while the cat fold barely moves, so the bistable window closes at a **cusp** $(\kappa_c, g_{2,c})$. Beyond it the cat deforms continuously into the satellite.
- For each $\kappa_b$, the satellite is taken at 120 MHz (it exists there for every $\kappa_b$, continued in $\kappa_b$), and the branch is followed by arclength towards lower $g_2$. Every turning point of $s$ along the way is recorded: the satellite fold (min), then the cat fold (max).
- The arclength step is capped at 0.002 near the cusp, so that the two turning points of a tiny S are not stepped over together.
- Near a cusp, the distance between the two folds scales as $(\kappa_c - \kappa_b)^{3/2}$. $\kappa_c$ is therefore estimated from a linear fit of $(\Delta g_2)^{2/3}$ against $\kappa_b$ over the last points.

In [ ]:
def S_turning_points(kfac, z_hi, s_hi, r=r0, ds_max=0.01, nmax=8000):
    """arclength along the S from the satellite at s_hi towards lower g2; returns [(g2 MHz, 'min'/'max', memory phase/pi)]"""
    hbk = HB(N_H, kfac)
    def F(y):
        hbk.set_s(y[-1], r); return hbk.residual(y[:-1])
    def dF(y):
        hbk.set_s(y[-1], r); Jz, r_ = hbk.jacobian(y[:-1]), hbk.residual(y[:-1])
        h = 1e-7
        hbk.set_s(y[-1] + h, r)
        return np.column_stack([Jz, (hbk.residual(y[:-1]) - r_)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dF(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and t[-1] > 0):
            t = -t
        return t
    y = np.append(z_hi, s_hi); t = tangent(y); ds = min(0.003, ds_max); hist = [y.copy()]; turns = []
    for _ in range(nmax):
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        if not sol.success:
            ds /= 2
            if ds < 1e-7:
                break
            continue
        y = sol.x; t_new = tangent(y, t); hist.append(y.copy())
        if np.sign(t_new[-1]) != np.sign(t[-1]):
            c = np.polyfit([-1, 0, 1], np.array(hist)[-3:, -1], 2)
            turns.append((MHz(np.polyval(c, -c[1]/(2*c[0]))*G0), "min" if t[-1] < 0 else "max", memory_phase(y[:-1])))
        t = t_new; ds = min(1.2*ds, ds_max)
        if len(turns) == 2 or MHz(y[-1]*G0) < 10:   # both folds found, or down on the cat at low g2 (no S)
            break
    return turns

# satellite at 120 MHz (kappa_b x 1), from the 60 MHz one
s_hi = 120/MHz(G0)
z_hi = z_sat60.copy()
for s_new in np.linspace(s_seed, s_hi, 25)[1:]:
    hb.set_s(s_new, r0); z_hi = hb.solve(z_hi)

k_list = [1, 2, 3, 4, 5, 6, 7, 8, 8.5, 9, 9.25, 9.5, 9.6, 9.65, 9.7, 9.75, 9.8, 10, 11]
cusp_scan, k_prev, z_k = [], 1.0, z_hi.copy()
for kfac in k_list:
    for kk in np.linspace(k_prev, kfac, 1 + int(np.ceil(abs(kfac - k_prev)/0.1)))[1:]:
        h_ = HB(N_H, kk); h_.set_s(s_hi, r0); z_k = h_.solve(z_k)
    k_prev = kfac
    turns = S_turning_points(kfac, z_k, s_hi, ds_max=0.01 if kfac < 9 else 0.002)
    g_sat = next((g for g, kind, _ in turns if kind == "min"), np.nan)
    g_cat = next((g for g, kind, _ in turns if kind == "max"), np.nan)
    cusp_scan.append((kfac, g_sat, g_cat))
    print(f"kappa_b x {kfac:5.2f} (kappa_b/2pi = {kfac*96.15:6.1f} MHz, GKSL): satellite fold {g_sat:7.2f} MHz, cat fold {g_cat:7.2f} MHz"
          + ("" if np.isfinite(g_sat) else "   -> no S"))
cusp_scan = np.array(cusp_scan)

# kappa_c from (Delta g2)^(2/3) linear in kappa_b (cusp normal form), last points with an S
ok = np.isfinite(cusp_scan[:, 1]) & (cusp_scan[:, 0] >= 8.5)
gap = cusp_scan[ok, 2] - cusp_scan[ok, 1]
pf = np.polyfit(cusp_scan[ok, 0], gap**(2/3), 1)
k_c = -pf[1]/pf[0]
g_c = np.polyval(np.polyfit(cusp_scan[ok, 0], (cusp_scan[ok, 1] + cusp_scan[ok, 2])/2, 2), k_c)
print(f"\ncusp: kappa_c = {k_c:.3f} kappa_b (kappa_b/2pi = {k_c*96.15:.0f} MHz in the GKSL convention), g2_c/2pi = {g_c:.1f} MHz, "
      f"g2_c/kappa_c = {g_c/(k_c*96.15):.3f}; residual of the 2/3-power fit: {np.std(np.polyval(pf, cusp_scan[ok, 0]) - gap**(2/3)):.1e}")

fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
ax.plot(cusp_scan[:, 0], cusp_scan[:, 1], "o-", ms=2, color=Perso_palette_float[5], label="satellite fold")
ax.plot(cusp_scan[:, 0], cusp_scan[:, 2], "o-", ms=2, color=Perso_palette_float[2], label="cat fold")
ax.plot(k_c, g_c, "k*", ms=6, label="cusp")
ax.fill_between(cusp_scan[:, 0], cusp_scan[:, 1], cusp_scan[:, 2], color="0.85", label="cat and satellite coexist")
ax.set_xlabel(r"$\kappa_b/\kappa_b^{(0)}$  ($\kappa_b^{(0)}/2\pi = 96$ MHz)"); ax.set_ylabel(r"$g_2/2\pi$ (MHz)")
ax.legend(fontsize=5)
ax = axes[1]
ax.plot(cusp_scan[ok, 0], gap**(2/3), "o", ms=3, color="k")
kk = np.linspace(cusp_scan[ok, 0].min(), k_c, 50)
ax.plot(kk, np.polyval(pf, kk), "--", color="gray")
ax.axvline(k_c, ls=":", color="k", lw=0.8)
ax.set_xlabel(r"$\kappa_b/\kappa_b^{(0)}$"); ax.set_ylabel(r"$(\Delta g_2/2\pi\,[\mathrm{MHz}])^{2/3}$")
plt.tight_layout()
plt.savefig(FIG_DIR + "cusp_kappa_b_20_10_2026.pdf")
plt.show()

### Limit cycles below and beyond the cusp
The period-1 branch is followed from the satellite at 120 MHz down to 8 MHz, at three values of $\kappa_b$:
- $\kappa_b^{(0)}$: the S of `Hill_method_cat_fold`;
- $8\,\kappa_b^{(0)}$: a narrow S, just below the cusp;
- $11\,\kappa_b^{(0)}$: beyond the cusp, $\kappa_c = 9.7\,\kappa_b^{(0)}$.

Stability comes from the monodromy matrix. In the left column, solid is stable and dashed unstable.
The middle and right columns show orbits along the branch, coloured by $g_2$, in the rotating frame of `limit_cycles.ipynb` ($a_R = i\,a\,e^{i\tau}$, $b_R = b\,e^{2i\tau}$), each with its partner $a\to-a$ in a lighter shade.

**What "beyond the cusp" means.** Below $\kappa_c$, at a given $g_2$ inside the S, there are three period-1 orbits (cat, saddle, satellite), each with its partner, and two of them are stable: the system is bistable.
Beyond $\kappa_c$ there is **one** period-1 orbit (plus its partner) at every $g_2$. The cat and the satellite are then no longer two distinct objects that coexist.
"Cat" and "satellite" only name the two ends of this single branch: cat-like at low $g_2$, satellite-like towards high $g_2$, with no jump and no fold in between.
This is the same as a Duffing oscillator above its critical damping: the response is single-valued, and the phase turns gradually instead of jumping.
The pair $\pm a$ is kept, so a qubit can still be encoded on it.

In [ ]:
def branch_from(kfac, z_start, s_start, g_min=8.0, ds_max=0.01):
    # arclength from (z_start, s_start) towards lower g2, down to g_min (MHz); returned from low g2 to s_start
    hbk = HB(N_H, kfac)
    def F(y):
        hbk.set_s(y[-1], r0); return hbk.residual(y[:-1])
    def dF(y):
        hbk.set_s(y[-1], r0); Jz, r_ = hbk.jacobian(y[:-1]), hbk.residual(y[:-1])
        h = 1e-7
        hbk.set_s(y[-1] + h, r0)
        return np.column_stack([Jz, (hbk.residual(y[:-1]) - r_)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dF(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and t[-1] > 0):
            t = -t
        return t
    y = np.append(z_start, s_start); t = tangent(y); ds = 0.003; hist = [y.copy()]
    for _ in range(10000):
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        if not sol.success:
            ds /= 2
            if ds < 1e-7:
                break
            continue
        y = sol.x; t = tangent(y, t); hist.append(y.copy()); ds = min(1.2*ds, ds_max)
        if MHz(y[-1]*G0) < g_min:
            break
    return np.array(hist[::-1])

def rotating_frame(z, t):
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, t)
    return 1j*(x_a + 1j*v_a)/(np.sqrt(2)*phi_a)*np.exp(1j*t), (x_b + 1j*v_b/2)/(np.sqrt(2)*phi_b)*np.exp(2j*t)

t_orb = np.linspace(0, 2*np.pi, 300)
beyond = {}
z_k, k_prev = z_hi.copy(), 1.0
for kfac in (1.0, 8.0, 11.0):
    for kk in np.linspace(k_prev, kfac, 1 + int(np.ceil(abs(kfac - k_prev)/0.1)))[1:]:
        h_ = HB(N_H, kk); h_.set_s(s_hi, r0); z_k = h_.solve(z_k)
    k_prev = kfac
    br = branch_from(kfac, z_k, s_hi)
    hbk = HB(N_H, kfac)
    idx = np.arange(0, len(br), 3)
    mu = []
    for i in idx:
        hbk.set_s(br[i, -1], r0); mu.append(max_abs_mu(hbk, br[i, :-1]))
    a2 = np.array([np.mean(np.abs(rotating_frame(br[i, :-1], t_orb)[0])**2) for i in idx])
    beyond[kfac] = dict(branch=br, idx=idx, g2=MHz(br[idx, -1]*G0), mu=np.array(mu), a2=a2,
                        phase=np.array([memory_phase(br[i, :-1]) for i in idx]))
    g2b = beyond[kfac]["g2"]; st = beyond[kfac]["mu"] < 1
    n_turn = np.sum(np.diff(np.sign(np.diff(g2b))) != 0)
    print(f"kappa_b x {kfac:4.1f}: {len(br)} points, {n_turn} turning point(s) of g2; "
          f"unstable for g2 in {np.round(np.sort(g2b[~st])[[0, -1]], 1) if (~st).any() else 'none'} MHz; "
          f"memory phase {beyond[kfac]['phase'][0]:+.3f} pi at {g2b[0]:.0f} MHz -> {beyond[kfac]['phase'][-1]:+.3f} pi at {g2b[-1]:.0f} MHz")

import matplotlib as mpl
cmap_g2 = plt.get_cmap("viridis"); norm_g2 = mpl.colors.Normalize(0, 120)
fig, axes = plt.subplots(3, 3, figsize=(3*figsize_car[0], 3*figsize_car[1]))
for row, kfac in enumerate((1.0, 8.0, 11.0)):
    d = beyond[kfac]; st = d["mu"] < 1
    ax = axes[row, 0]
    ax.plot(d["g2"], np.where(st, d["a2"], np.nan), "-", color="k", lw=1)
    ax.plot(d["g2"], np.where(st, np.nan, d["a2"]), "--", color=Perso_palette_float[4], lw=1)
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\langle|a_R|^2\rangle$")
    ax.set_title(rf"$\kappa_b = {kfac:g}\,\kappa_b^{{(0)}}$" + (" (beyond the cusp)" if kfac > 9.7 else " (S-shaped)"), fontsize=8)
    ax.set_xlim(0, 125)
    # orbits along the branch, evenly spaced in arclength
    br = d["branch"]
    for i in np.linspace(0, len(br) - 1, 14).astype(int):
        hbk = HB(N_H, kfac); hbk.set_s(br[i, -1], r0)
        stable = max_abs_mu(hbk, br[i, :-1]) < 1
        a_R, b_R = rotating_frame(br[i, :-1], t_orb)
        col = cmap_g2(norm_g2(MHz(br[i, -1]*G0)))
        for sign, alpha in ((1, 1.0), (-1, 0.3)):
            axes[row, 1].plot(sign*a_R.real, sign*a_R.imag, "-" if stable else "--", color=col, lw=0.8, alpha=alpha)
        axes[row, 2].plot(b_R.real, b_R.imag, "-" if stable else "--", color=col, lw=0.8)
    zoom = 1.0 if kfac == 1.0 else 0.4                     # the branch is much smaller at large kappa_b
    for ax, lab, lim in ((axes[row, 1], "a", 9.5*zoom), (axes[row, 2], "b", 5.0*zoom)):
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        ax.set_xlabel(rf"Re ${lab}_R$"); ax.set_ylabel(rf"Im ${lab}_R$")
axes[0, 0].plot([], [], "-", color="k", label="stable"); axes[0, 0].plot([], [], "--", color=Perso_palette_float[4], label="unstable (saddle or period-doubled)")
axes[0, 0].legend(fontsize=5)
fig.subplots_adjust(hspace=0.45, wspace=0.4)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_g2, cmap=cmap_g2), ax=axes[:, 1:], label=r"$g_2/2\pi$ (MHz)", shrink=0.6)
plt.savefig(FIG_DIR + "limit_cycles_beyond_cusp_20_10_2026.pdf", bbox_inches="tight")
plt.show()

## Beyond the cusp at large $g_2$: period doubling or Neimark–Sacker? (22/10/2026)
At $\kappa_b^{(0)}$ the satellite end of the S period-doubles at 75.3 MHz and the doubled orbit then undergoes a Neimark–Sacker bifurcation (`floquet_rescaling_kappa_b.ipynb`).
The question here is whether the single period-1 branch beyond the cusp ($\kappa_b > \kappa_c$) goes through the same sequence as $g_2$ grows above 120 MHz.
1. **Period-doubling bubble of the satellite vs $\kappa_b$.** For each $\kappa_b$ the period-1 orbit at 120 MHz is followed down to the satellite fold (or to 20 MHz beyond the cusp) and up to $g_2^{\max}$.
   A multiplier crossing $-1$ is a period doubling.
2. **Formal continuation above $g_2^{\max}$.** The physical range ends at $\varepsilon_p = \pi/2$, $g_2^{\max} = E_J\varphi_a^2\varphi_b = 137.4$ MHz.
   The equations depend on $\varepsilon_p$ only through $s = \sin\varepsilon_p$ ($\beta'\propto s$, $\varepsilon_d = r\,sG_0$), so continuing to $s > 1$ is the same model with a larger $E_J\varphi_a^2\varphi_b$, not reachable with this circuit.
   There the first loss of stability is located by bisection on $\max|\mu| = 1$ and classified by the crossing multiplier: $-1$ period doubling, $+1$ fold, complex pair Neimark–Sacker.
3. **Torus check** just above the Neimark–Sacker point, by time integration (supercritical or subcritical).
4. **Other attractors** at $11\,\kappa_b^{(0)}$, 130 MHz: random initial states.

All multipliers come from the monodromy matrix (`mu_monodromy` returns the four of them; `max_abs_mu` above returns only the largest modulus).

In [ ]:
def mu_monodromy(hbk, z):
    # the four Floquet multipliers of the HB orbit z (variational equation over one period)
    Minv, base = np.linalg.inv(M), np.outer(hbk.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hbk.N_H, np.atleast_1d(t))
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hbk.C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

def solve_robust(hbk, z):
    try:
        return hbk.solve(z)
    except RuntimeError:
        return hbk.solve(z, tol=1e-9)

def orbit_at_kappa(kfac):
    # period-1 orbit at 120 MHz for kappa_b x kfac, continued in kappa_b from the satellite z_hi (kappa_b x 1)
    z = z_hi.copy()
    for kk in np.linspace(1.0, kfac, 1 + int(np.ceil(abs(kfac - 1)/0.1)))[1:]:
        h_ = HB(N_H, kk); h_.set_s(s_hi, r0); z = h_.solve(z)
    return z

def level_crossings(g, y, level):
    # g2 values where y crosses level (linear interpolation between grid points)
    return [g[j] + (level - y[j])*(g[j + 1] - g[j])/(y[j + 1] - y[j]) for j in np.where(np.diff(np.sign(y - level)) != 0)[0]]

k_scan = [1, 2, 3, 4, 5, 5.25, 5.5, 6, 7, 8, 9, 9.5, 10, 11, 12, 15, 20]
p1 = {}
for kfac in k_scan:
    hbk, z0 = HB(N_H, kfac), orbit_at_kappa(kfac)
    rows = []
    for path in (np.arange(s_hi, 20/MHz(G0), -0.004), np.linspace(s_hi, 1.0, 31)):
        z = z0.copy()
        for s in path:
            hbk.set_s(s, r0)
            try:
                zn = hbk.solve(z)
            except RuntimeError:
                break                                        # satellite fold (below the cusp)
            if abs(memory_phase(zn) - memory_phase(z)) > 0.05:
                break                                        # jumped to the cat side: past the satellite fold
            z = zn
            mu = mu_monodromy(hbk, z)
            re = mu[np.abs(mu.imag) < 1e-9].real
            rows.append((MHz(s*G0), np.abs(mu).max(), re.min() if len(re) else np.nan))
    g, mumax, remin = np.array(sorted(set(rows))).T
    pd = level_crossings(g, np.where(np.isfinite(remin), remin, 0.0), -1.0)   # a -1 crossing needs a real multiplier
    has_real = np.isfinite(remin).any()                  # at large kappa_b the multipliers can stay complex on the whole branch
    p1[kfac] = dict(g2=g, mumax=mumax, remin=remin, pd=pd, re_min=np.nanmin(remin) if has_real else np.nan,
                    g_min=g[np.nanargmin(remin)] if has_real else np.nan)
    unst = g[mumax > 1]
    print(f"kappa_b x {kfac:5.2f} ({kfac*96.15:5.0f} MHz GKSL): {g[0]:5.1f}-{g[-1]:5.1f} MHz, "
          + (f"most negative real mu {p1[kfac]['re_min']:+.3f} at {p1[kfac]['g_min']:5.1f} MHz, " if has_real else "no real multiplier, ")
          + (f"PD bubble {pd[0]:.1f}-{pd[-1]:.1f} MHz" if len(pd) else "no period doubling")
          + (f", unstable on {unst.min():.1f}-{unst.max():.1f} MHz" if len(unst) else ", stable everywhere"))

# kappa_b where the bubble closes: the most negative multiplier reaches -1
k_arr, m_arr = np.array(k_scan, float), np.array([p1[k]["re_min"] for k in k_scan])
ok = np.isfinite(m_arr)
k_pd = level_crossings(k_arr[ok], m_arr[ok], -1.0)[0]
g_pd = np.interp(k_pd, k_arr[ok], np.array([p1[k]["g_min"] for k in k_scan])[ok])
print(f"\nperiod-doubling bubble closes at kappa_b = {k_pd:.2f} kappa_b0 ({k_pd*96.15:.0f} MHz GKSL), g2/2pi = {g_pd:.0f} MHz; cusp at {k_c:.2f} kappa_b0")

In [ ]:
def first_instability(kfac, ds=0.002, s_end=2.0):
    # natural continuation in s = sin(eps_p) from 120 MHz upward (formally past s = 1) until max|mu| > 1, then bisection
    hbk, z = HB(N_H, kfac), orbit_at_kappa(kfac)
    s_prev = s_hi
    for s in np.arange(s_hi + ds, s_end, ds):
        hbk.set_s(s, r0); zn = solve_robust(hbk, z)
        if np.abs(mu_monodromy(hbk, zn)).max() > 1:
            lo, hi, z_lo = s_prev, s, z
            for _ in range(30):
                mid = (lo + hi)/2
                hbk.set_s(mid, r0); z_mid = solve_robust(hbk, z_lo)
                if np.abs(mu_monodromy(hbk, z_mid)).max() > 1:
                    hi = mid
                else:
                    lo, z_lo = mid, z_mid
            hbk.set_s(hi, r0); z_c = solve_robust(hbk, z_lo)
            mu = mu_monodromy(hbk, z_c); m = mu[np.argmax(np.abs(mu))]
            kind = "Neimark-Sacker" if abs(m.imag) > 1e-6 else ("period doubling" if m.real < 0 else "fold (+1)")
            return dict(g2=MHz(hi*G0), s=hi, kind=kind, arg=abs(np.angle(m))/np.pi, z=z_c, tail=np.max(np.abs(z_c[-4*n:])))
        z, s_prev = zn, s
    return dict(g2=np.nan, s=np.nan, kind="none", arg=np.nan, z=None, tail=np.nan)

k_formal = [1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 15, 20]
formal = {}
for kfac in k_formal:
    formal[kfac] = first_instability(kfac)
    f_ = formal[kfac]
    print(f"kappa_b x {kfac:5.2f} ({kfac*96.15:5.0f} MHz GKSL): first instability above 120 MHz: {f_['kind']} at g2/2pi = {f_['g2']:6.2f} MHz "
          f"(s = sin eps_p = {f_['s']:.4f}), arg mu = {f_['arg']:.4f} pi, HB tail {f_['tail']:.0e}")

In [ ]:
# torus check at kappa_b x 11: time integration from the HB orbit + 1e-4, 400/kappa_b of transient, then 300 stroboscopic points
kfac = 11.0
g_NS, s_cur, z = formal[kfac]["g2"], formal[kfac]["s"], formal[kfac]["z"].copy()
hbk = HB(N_H, kfac)
T_tr = 2*np.pi*int(round(400*w_a/(kfac*kappa_b)/(2*np.pi)))
torus = []
for dg in (-0.5, 0.1, 0.2, 0.4, 0.6, 0.8):
    s_t = (g_NS + dg)/MHz(G0)
    for s in np.linspace(s_cur, s_t, 1 + int(np.ceil(abs(s_t - s_cur)/0.001)))[1:]:
        hbk.set_s(s, r0); z = solve_robust(hbk, z)
    s_cur = s_t
    y_p, f = strobe(z, N_H), full_rhs(s_t, r0, kfac)
    y = solve_ivp(f, (0, T_tr), y_p + 1e-4, method="DOP853", rtol=1e-11, atol=1e-13, t_eval=[T_tr]).y[:, -1]
    tt = T_tr + 2*np.pi*np.arange(301)
    P = solve_ivp(f, (T_tr, tt[-1]), y, method="DOP853", rtol=1e-11, atol=1e-13, t_eval=tt).y.T
    d = np.linalg.norm(P - y_p, axis=1)
    ang = np.unwrap(np.angle((P[:, 0] - y_p[0]) + 1j*(P[:, 2] - y_p[2])))
    rot = abs(ang[-1] - ang[0])/(2*np.pi*(len(ang) - 1))
    torus.append((dg, d.max()))
    print(f"g2 = g2_NS {dg:+.2f} MHz: stroboscopic distance to the period-1 orbit {d.min():.1e} .. {d.max():.1e}"
          + (f", rotation number {rot:.4f} (arg mu_NS/2pi = {formal[kfac]['arg']/2:.4f})" if 1e-6 < d.max() < 0.1 else ""))
torus = np.array(torus); m = (torus[:, 0] > 0) & (torus[:, 1] < 0.1)
print(f"torus size ~ (g2 - g2_NS)^{np.polyfit(np.log(torus[m, 0]), np.log(torus[m, 1]), 1)[0]:.2f}")

In [ ]:
# other attractors at kappa_b x 11, 130 MHz: where do generic initial states end?
kfac, g_test = 11.0, 130.0
hbk, z = HB(N_H, kfac), orbit_at_kappa(kfac)
for s in np.linspace(s_hi, g_test/MHz(G0), 15)[1:]:
    hbk.set_s(s, r0); z = hbk.solve(z)
s_t = g_test/MHz(G0)
y_p, y_m = strobe(z, N_H, 0.0), strobe(z, N_H, np.pi)
f = full_rhs(s_t, r0, kfac)
T_tr = 2*np.pi*int(round(2000*w_a/(kfac*kappa_b)/(2*np.pi)))
alpha = np.sqrt(r0/2)
rng = np.random.default_rng(1)
starts = {"branch orbit + 1e-3 kick": y_p + 1e-3, "origin": np.zeros(4), "cat start a = i|alpha|, b = 0": np.array([0, 0, np.sqrt(2)*phi_a*alpha, 0])}
for j in range(8):
    starts[f"random {j}"] = rng.normal(size=4)*np.sqrt(2)*np.array([3*phi_a, 2*phi_b, 3*phi_a, 4*phi_b])
a_R_branch = rotating_frame(z, np.linspace(0, 2*np.pi, 400, endpoint=False))[0]
print(f"branch orbit: <a_R> = {a_R_branch.mean().real:+.2f}{a_R_branch.mean().imag:+.2f}i (partner: opposite), <|a_R|> = {np.abs(a_R_branch).mean():.2f}, "
      f"max|mu| = {np.abs(mu_monodromy(hbk, z)).max():.3f}")
t1 = np.linspace(0, 2*np.pi, 400, endpoint=False)
for name, y0 in starts.items():
    y = solve_ivp(f, (0, T_tr), y0, method="DOP853", rtol=1e-10, atol=1e-12, t_eval=[T_tr]).y[:, -1]
    Y = solve_ivp(f, (T_tr, T_tr + 2*np.pi), y, method="DOP853", rtol=1e-10, atol=1e-12, t_eval=np.append(T_tr + t1, T_tr + 2*np.pi)).y
    a_R = 1j*(Y[0, :-1] + 1j*Y[2, :-1])/(np.sqrt(2)*phi_a)*np.exp(1j*t1)             # rotating frame over one period
    d = min(np.linalg.norm(y - y_p), np.linalg.norm(y - y_m))
    print(f"{name:30s}: |x(T + 2pi) - x(T)| = {np.linalg.norm(Y[:, -1] - y):.0e} (period 1), distance to the branch pair {d:.0e}, "
          f"<a_R> = {a_R.mean().real:+.2f}{a_R.mean().imag:+.2f}i, <|a_R|> = {np.abs(a_R).mean():.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(1.5*figsize[0], figsize[1]))
okc = np.isfinite(cusp_scan[:, 1])
ax.fill_between(cusp_scan[okc, 0], cusp_scan[okc, 1], cusp_scan[okc, 2], color="0.88", lw=0, label="cat and satellite coexist")
ax.plot(cusp_scan[okc, 0], cusp_scan[okc, 1], color=Perso_palette_float[5], lw=1, label="satellite fold")
ax.plot(cusp_scan[okc, 0], cusp_scan[okc, 2], color=Perso_palette_float[2], lw=1, label="cat fold")
ax.plot(k_c, g_c, "k*", ms=6, label="cusp")
kb = [k for k in k_scan if len(p1[k]["pd"]) >= 2]
ax.fill_between(kb + [k_pd], [p1[k]["pd"][0] for k in kb] + [g_pd], [p1[k]["pd"][-1] for k in kb] + [g_pd],
                color=Perso_palette_float[4], alpha=0.35, lw=0, label="satellite period-doubled")
ax.plot(k_formal, [formal[k]["g2"] for k in k_formal], "o--", ms=2, color="k", lw=0.8, label=r"Neimark–Sacker of the period-1 orbit (formal, $s > 1$)")
ax.axhline(MHz(G0), color="gray", lw=0.8, label=r"$g_2^{\max} = E_J\varphi_a^2\varphi_b$")
ax.set_xlim(0, 20.5); ax.set_ylim(20, 220)
ax.set_xlabel(r"$\kappa_b/\kappa_b^{(0)}$  ($\kappa_b^{(0)}/2\pi = 96$ MHz)"); ax.set_ylabel(r"$g_2/2\pi$ (MHz)")
ax.legend(fontsize=5, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.savefig(FIG_DIR + "beyond_cusp_bifurcations_22_10_2026.pdf", bbox_inches="tight")
plt.show()

with open(OUT_DIR + "beyond_cusp_bifurcations_22_10_2026.json", "w") as f:
    json.dump({"description": "Period-1 branch at eps_d = 10.12 g2 vs kappa_b (Hill_method_satellite_fold.ipynb): satellite period-doubling bubble, "
                              "and first instability of the formal continuation above g2_max (s = sin eps_p > 1)",
               "kappa_b0_code": kappa_b, "N_H": N_H, "g2_max_MHz": MHz(G0),
               "kappa_over_kappa0": k_scan,
               "pd_bubble_MHz": [p1[k]["pd"] for k in k_scan],
               "most_negative_real_mu": m_arr.tolist(),
               "kappa_pd_close": k_pd, "g2_pd_close_MHz": g_pd, "kappa_cusp": k_c, "g2_cusp_MHz": g_c,
               "formal": {str(k): {kk: formal[k][kk] for kk in ("g2", "s", "kind", "arg")} for k in k_formal}}, f)

**Reading.**
- **No period doubling beyond the cusp.** The period-doubling bubble of the satellite (75.5–103.5 MHz at $\kappa_b^{(0)}$) shrinks as $\kappa_b$ grows and closes at $\kappa_b \simeq 5.4\,\kappa_b^{(0)}$ ($\approx 520$ MHz GKSL, $g_2 \approx 99$ MHz), well before the cusp at $9.7\,\kappa_b^{(0)}$.
  The period-2 orbit, and hence its Neimark–Sacker, exist only inside the bubble, so neither is left beyond the cusp.
  What remains is a real multiplier that dips towards $-1$ (to $-0.79$ at 121 MHz for $10\,\kappa_b^{(0)}$, $-0.76$ for $11\,\kappa_b^{(0)}$) and recedes as $\kappa_b$ grows.
- **Stable up to $g_2^{\max}$.** For $\kappa_b \ge 6\,\kappa_b^{(0)}$ the period-1 orbit is linearly stable on the whole physical range, up to $g_2^{\max} = 137.4$ MHz (at $11\,\kappa_b^{(0)}$, $\max|\mu| \le 0.76$ above 120 MHz; $N_H = 24$ gives the same multipliers to 4 digits).
- **Formal continuation: a Neimark–Sacker of the period-1 orbit itself, never a period doubling.** The first instability is a complex pair leaving the unit circle at $\arg\mu \simeq \pm0.61$–$0.69\,\pi$,
  at $g_2^{NS}/2\pi$ = 142 MHz ($\kappa_b^{(0)}$), 151 MHz ($6\,\kappa_b^{(0)}$), 164 MHz ($10\,\kappa_b^{(0)}$), 168.4 MHz ($11\,\kappa_b^{(0)}$), 186 MHz ($15\,\kappa_b^{(0)}$), 209 MHz ($20\,\kappa_b^{(0)}$).
  The line is smooth through the cusp, so it belongs to the satellite end of the branch, not to the cusp. It is always above $g_2^{\max}$ (by 3% at $\kappa_b^{(0)}$): it would need a larger $E_J\varphi_a^2\varphi_b$.
- **The Neimark–Sacker is supercritical.** At $11\,\kappa_b^{(0)}$ a small torus appears just above $g_2^{NS}$, with a rotation number that starts at $\arg\mu_{NS}/2\pi$ and a size growing as $(g_2 - g_2^{NS})^{\approx 1/2}$ (fit exponent 0.59).
  It is lost between $+0.6$ and $+0.8$ MHz, while its rotation number approaches $1/3$; the trajectory then ends on another period-1 orbit.
- **Other attractors.** Beyond the cusp there is a single period-1 orbit *on the S branch* (plus its partner $a\to-a$), not a single attractor in phase space. At $11\,\kappa_b^{(0)}$ and 130 MHz:
  - the kicked branch orbit, the origin, the cat start $a = i|\alpha|$, $b = 0$ and 3 of the 8 random starts end on the branch pair ($\langle|a_R|\rangle = 2.45$; the origin and the cat start land on the partner);
  - 4 random starts end on another, larger pair ($\langle|a_R|\rangle \approx 5.8$, with a much larger buffer);
  - 1 ends on a self-symmetric orbit that circles the origin in the rotating frame ($\langle a_R\rangle = 0$, $\langle|a_R|\rangle \approx 5.6$).

  None is chaotic. This does not affect the branch orbit: it is linearly stable, so a trajectory started on it or near it stays on it. It only means that its basin is not the whole phase space.

## The band of the satellite-start map where the trajectory ends on a cat
At $\varepsilon_d/2g_2 \approx 5.05$ the map has a satellite seed from 35.3 MHz (the analytic boundary on the map grid) but a satellite end state only from 49.7 MHz.
- Below the HB fold, 40.4 MHz, the analytic satellite does not exist: the quadratic theory puts its fold 15% too low.
- Between 40.4 and about 50 MHz the satellite exists and is stable. Below, the trajectory is integrated for $300/\kappa_b$ from two starts:
  - the **analytic seed** (the map's initial state);
  - the **exact HB satellite** plus a $10^{-4}$ kick.

  "Distance" is to the nearer of the two satellites of the pair (stroboscopic, $\tau = 2\pi k$), in $x^\lambda$ units.

In [ ]:
T_end = 2*np.pi*int(round(300*w_a/kappa_b/(2*np.pi)))
z = z_sat60.copy(); s_cur = s_seed
for g in (60.0, 48.0, 45.0, 42.0):
    for s_new in np.linspace(s_cur, g/MHz(G0), 8)[1:]:
        hb.set_s(s_new, r0); z = hb.solve(z)
    s_cur = g/MHz(G0)
    y_p, y_m = strobe(z, N_H, 0.0), strobe(z, N_H, np.pi)
    S = satellite_hb(s_cur, r0)
    y_seed = seed_state(S)
    dist0 = min(np.linalg.norm(y_seed - y_p), np.linalg.norm(y_seed - y_m))
    out = []
    for y0 in (y_seed, y_p + 1e-4):
        sol = solve_ivp(full_rhs(s_cur, r0), (0, T_end), y0, method="DOP853", rtol=1e-10, atol=1e-12, t_eval=[T_end])
        out.append(min(np.linalg.norm(sol.y[:, -1] - y_p), np.linalg.norm(sol.y[:, -1] - y_m)))
    print(f"g2/2pi = {g:4.0f} MHz: max|mu| = {max_abs_mu(hb, z):.4f}; analytic seed is {dist0:.2f} from the satellite; "
          f"after 300/kappa_b: from the seed {out[0]:.1e} ({'satellite' if out[0] < 1e-3 else 'not on the satellite'}), from the HB orbit {out[1]:.1e}")

## Figure: fold lines on the satellite-start map
The background is the end state of the satellite-start map (`end_state_satellite_start_14_10_2026.json`): 1 satellite, 2 cat, 3 chaotic, blank where there is no analytic seed.
The lines are the satellite fold (HB, reduced full cosine, reduced quadratic = analytic boundary, closed form) and the cat fold (`cat_fold_line_19_10_2026.json`).

In [ ]:
E = np.array(json.load(open(OUT_DIR + "end_state_satellite_start_14_10_2026.json")), dtype=float)        # (eps_p, alpha^2)
g2_map = MHz(np.array(json.load(open(OUT_DIR + "g2_list_satellite_start_12_10_2026.json"))))
alpha_sq_map = np.array(json.load(open(OUT_DIR + "alpha_sq_list_satellite_start_12_10_2026.json")))
cat = json.load(open(OUT_DIR + "cat_fold_line_19_10_2026.json"))
cat_r = np.array(cat["eps_d_over_g2"]); cat_g = np.array([np.nan if v is None else v for v in cat["g2_fold_MHz"]], float)

from matplotlib.colors import ListedColormap, to_hex
cmap = ListedColormap([to_hex(Perso_palette_float[5]) + "55", to_hex(Perso_palette_float[3]) + "55", to_hex(Perso_palette_float[4]) + "55"])
fig, ax = plt.subplots(figsize=(1.3*figsize[0], 1.3*figsize[1]))
ax.pcolormesh(g2_map, alpha_sq_map, E.T, cmap=cmap, vmin=0.5, vmax=3.5, shading="nearest", rasterized=True)
ax.plot(g2_sat_fold, ratio_list/2, color="k", lw=1.5, label="satellite fold (Hill)")
ax.plot(red_full, ratio_list/2, "--", color="k", lw=0.8, label="satellite fold, reduced (full cosine)")
ax.plot(red_quad, ratio_list/2, ":", color="k", lw=1, label="analytic boundary = reduced (quadratic)")
ax.plot(closed, ratio_list/2, "-.", color="gray", lw=0.8, label=r"closed form $\propto\sqrt{\kappa_b}/|\alpha|$")
ax.plot(cat_g, cat_r/2, color=Perso_palette_float[2], lw=1.5, label="cat fold (Hill)")
ax.set_xlim(g2_map[0], g2_map[-1]); ax.set_ylim(1, 6.75)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
ax.legend(fontsize=4.5, loc="upper center")
ax.set_title("background: end state from the satellite seed (olive: satellite, blue: cat, red: chaos)", fontsize=4.5)
plt.tight_layout()
plt.savefig(FIG_DIR + "satellite_fold_line_20_10_2026.pdf")
plt.show()

with open(OUT_DIR + "satellite_fold_line_20_10_2026.json", "w") as f:
    json.dump({"description": "Satellite fold (birth of the satellite with the saddle orbit), Hill_method_satellite_fold.ipynb",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H,
               "eps_d_over_g2": ratio_list.tolist(),
               "g2_fold_MHz_hill": g2_sat_fold.tolist(),
               "memory_phase_at_fold_over_pi": [sat_fold[r][1] for r in ratio_list],
               "g2_fold_MHz_reduced_full_cos": red_full.tolist(),
               "g2_fold_MHz_reduced_quad": red_quad.tolist(),
               "g2_MHz_analytic_existence_boundary": analytic.tolist(),
               "g2_MHz_closed_form": closed.tolist(),
               "kappa_check_MHz": {str(k_): v for k_, v in kappa_check.items()}}, f)

## Conclusions
1. **Satellite fold line.** The fold goes from $g_2/2\pi = 84.4$ MHz at $\varepsilon_d/g_2 = 2$ down to 33.6 MHz at $\varepsilon_d/g_2 = 13$ (40.4 MHz at 10.12).
   - In the form $g_2^\ast|\alpha|$ it varies between 84 and 91 MHz. The cat fold is at $g_2^\ast|\alpha| \approx 219$ MHz, 2.4–2.6 times higher at every ratio.
   - **The two fold lines never meet in the map: there is no cusp in the $(g_2, \varepsilon_d/2g_2)$ plane.** Cat and satellite coexist on a bistable window at every ratio.
   - The cusp comes from $\kappa_b$ (section "Targeting the cusp"). The satellite fold rises with $\kappa_b$, the cat fold only slowly, from 97.2 to 110.6 MHz.
     They merge at $\kappa_c = 9.70\,\kappa_b$ ($\kappa_b/2\pi \approx 933$ MHz in the GKSL convention), at $g_{2,c}/2\pi = 110.6$ MHz, i.e. $g_{2,c}/\kappa_c \approx 0.12$, for $\varepsilon_d = 10.12\,g_2$.
     The gap closes as $(\kappa_c - \kappa_b)^{3/2}$, as expected for a cusp. Beyond $\kappa_c$ there is no bistability: at each $g_2$ there is a single stable period-1 orbit (and its partner $a\to-a$), cat-like at low $g_2$ and satellite-like at high $g_2$, with no fold in between. Cat and satellite are then no longer distinct coexisting orbits (section \"Limit cycles below and beyond the cusp\").
     The power law $\kappa_b^{0.66}$ only holds near $\kappa_b^{(0)}$ (an extrapolation from it gave $\approx 4\kappa_b$, which is wrong).
2. **The analytic existence boundary is a fold.**
   - The boundary used for the seeds of the satellite-start map coincides, within 0.02 MHz, with the fold of the reduced system at quadratic order.
   - It is the phase-locking boundary $2\kappa_b'|y| = \varepsilon_d'$ (the two satellite roots merge, $\cos 2\theta \to 0$), not $x^2 \to 0$. At the HB fold the memory phase is $0.736$–$0.743\,\pi$, i.e. within $0.015\pi$ of quadrature.
   - It is, however, **8–15% too low**. The full cosine (reduced system, still 3 harmonics) brings the error down to 1.6–4.9%, with the exact phase at the fold.
3. **Closed form.** At leading order ($u = 0$, $\varepsilon_\text{eff} = 0$, $\bar y = 1/7\beta_\Sigma$):
   $$g_2^\ast|\alpha| = \varphi_a\sqrt{\frac{\kappa_b\,\omega_a}{7(2\varphi_a^2+\varphi_b^2)}}\quad\text{(code } \kappa_b\text{; } \kappa_b^\text{GKSL}/14 \text{ in place of } \kappa_b/7\text{)},$$
   that is 71 MHz/$|\alpha|$, 15–22% below Hill.
   - It gives the scalings $1/|\alpha|$ and $\sqrt{\kappa_b}$.
   - The exact exponent in $\kappa_b$ is 0.66 (Hill; 0.64 for the reduced full cosine, 0.50 at quadratic order). The difference comes from the full cosine.
4. **Band of the satellite-start map** ($\varepsilon_d/2g_2 \approx 5$: seed from 35 MHz, satellite end state from 50 MHz).
   - Up to the HB fold, 40.4 MHz, there is no satellite at all: the analytic boundary is 15% too low.
   - From 40.4 to about 50 MHz the satellite exists and is stable ($\max|\mu| \le 0.98$). The analytic seed is 1.1–1.4 away from it (in $x^\lambda$ units) and ends on the cat, while the exact orbit plus a kick stays on the satellite.
   - At 60 MHz the seed is 0.54 away and lands on the satellite.
   - So the lower edge of the satellite region of the map, at about $1.2\times$ the fold, measures the quality of the analytic seed, not where the satellite exists.
5. **Beyond the cusp at large $g_2$** (section "Beyond the cusp at large $g_2$").
   - The satellite's period-doubling bubble closes at $\kappa_b \simeq 5.4\,\kappa_b^{(0)}$, before the cusp ($9.7\,\kappa_b^{(0)}$). Beyond the cusp there is no period doubling and no period-2 orbit, hence no Neimark–Sacker of it.
   - The period-1 orbit is stable up to $g_2^{\max} = 137.4$ MHz for every $\kappa_b \ge 6\,\kappa_b^{(0)}$.
   - Continued formally above $g_2^{\max}$ ($\sin\varepsilon_p > 1$, i.e. a larger $E_J\varphi_a^2\varphi_b$), the period-1 orbit loses stability through a supercritical Neimark–Sacker bifurcation ($\arg\mu \approx 0.62\pi$), at 142 MHz for $\kappa_b^{(0)}$ up to 209 MHz for $20\,\kappa_b^{(0)}$ (168.4 MHz at $11\,\kappa_b^{(0)}$).
   - Beyond the cusp the branch orbit is unique on the S, but other stable period-1 orbits (larger amplitude) coexist with it in phase space; the origin and the cat start still end on the branch.
